In [1]:
using LowLevelFEM

structured_rect_mesh(lx=1.0, ly=0.04, y0=-0.02, n=2, order=2)
mat = Material("body"; E=1e6, ν=0.3, ρ=1000.0)
U = Field([mat]; type=:VectorField, dim=2, fieldName=:u)
b, D2 = 0.01, D(:PlaneStress, mat)
support = [BoundaryCondition("left"; field=U, ux=0.0, uy=0.0)]
u0 = initialDisplacement(U, "body"; ux=0.0, uy=0.0)
v0 = initialVelocity(U, "body"; vx=0.0, vy=(x,y,z)->0.25*x^2*(3-x)/2)
M = b*mat.ρ*∫(U⋅U)
K = b*∫(SymGrad(U)⋅D2⋅SymGrad(U))
#K = ∫(SymGrad(U)⋅D2⋅SymGrad(U); weight=b, csc_matrix=build_csc_pattern(U, U))
B0 = [1 0 0 0; 0 0 0 1; 0 1 1 0];

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 4 for different dependency version already loaded, 2 for different Julia build configuration, 3 for file size changed)
Precompiling packages...
 169.2 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 170 seconds. 106 already precompiled.


In [2]:
@inline function secant(A, state)
    #print(".")
    H = ∇(expandTo3D(state.fields[1]))
    F = tensorToMatrix(unitTensor(H) + H)
    B = [F[1,1] 0 F[2,1] 0; 0 F[1,2] 0 F[2,2];
         F[1,2] F[1,1] F[2,2] F[2,1]]
    b*∫(Grad(U)⋅B'⋅D2⋅((B+B0)/2)⋅Grad(U); gauss=4)
    #fill!(A.A.nzval, 0.0)
    #∫(Grad(U)⋅B'⋅D2⋅((B+B0)/2)⋅Grad(U); gauss=4, weight=b, csc_matrix=A.A)
end

secant (generic function with 1 method)

In [3]:
@time u, v = HHT(K, M, 0*u0, u0, v0, 401, 0.02;
    support, α=-0.05, modifyA0=secant,
    nonlinearoptions=(maxiter=50, abstol=1e-5),
    copyOperators=true)
showDoFResults(u; name="Nagy lengés — HHT", factor=1.0, visible=true)

 42.888375 seconds (176.66 M allocations: 26.480 GiB, 12.49% gc time, 34.61% compilation time)


0

In [4]:
@time begin
    H = ∇(expandTo3D(u))
    F = tensorToMatrix(unitTensor(H) + H)
    B = [F[1,1] 0 F[2,1] 0; 0 F[1,2] 0 F[2,2];
        F[1,2] F[1,1] F[2,2] F[2,1]]
    b*∫(Grad(U)⋅B'⋅D2⋅((B+B0)/2)⋅Grad(U); gauss=4)
end

  0.279930 seconds (501.57 k allocations: 189.115 MiB, 9.04% gc time, 51.46% compilation time)


sparse([1, 2, 9, 10, 107, 108, 411, 412, 415, 416  …  311, 312, 513, 514, 1001, 1002, 1005, 1006, 1009, 1010], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1010, 1010, 1010, 1010, 1010, 1010, 1010, 1010, 1010, 1010], [4615.384615384616, 1785.7142857142853, 189.2551892551896, 45.787545787545405, -3308.9133089133097, -183.15018315018122, -683.7606837606834, -45.78754578754568, 341.88034188034237, 183.15018315018222  …  1.2241798685863615e-11, -14261.29426129428, -3174.6031746031763, -2637.362637362624, -4.324647306930274e-12, -1562.8815628815682, -4.342837200965733e-12, -14261.294261294244, -6.5301719587296245e-12, 42197.80219780238], 1010, 1010)

In [8]:

iterations = zeros(Int, 400)

secant_counted = (A, state) -> begin
    if state.phase === :step
        iterations[state.step] = state.iteration
    end
    secant(A, state)
end

@time u, v = HHT(K, M, 0*u0, u0, v0, 401, 0.02;
    support, α=-0.05,
    modifyA0=secant_counted,
    nonlinearoptions=(maxiter=50, abstol=1e-7),
    copyOperators=false)

println("Total iterations: ", sum(iterations))
println("Mean iterations: ", sum(iterations)/400)
println("Maximum iterations: ", maximum(iterations))


 43.093005 seconds (219.90 M allocations: 35.003 GiB, 13.31% gc time, 3.54% compilation time)
Total iterations: 1912
Mean iterations: 4.78
Maximum iterations: 9


In [6]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
